In [5]:
import torch
import tiktoken

torch.manual_seed(123)
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()


tokenizer = tiktoken.get_encoding("gpt2")

In [9]:
ids = tokenizer.encode(raw_text)[3:6]
print([tokenizer.decode([i]) for i in ids])

[' always', ' thought', ' Jack']


In [11]:
torch.tensor(ids).shape

torch.Size([3])

In [15]:
embedding = torch.nn.Embedding(50257, 3)
words = embedding(torch.tensor(ids))
print(words)


tensor([[-0.2660, -1.2898, -0.3104],
        [ 1.4100,  0.8201,  0.0837],
        [ 1.6249,  2.9188,  0.0243]], grad_fn=<EmbeddingBackward0>)


In [20]:
W_x = torch.randn(4,3)
W_h = torch.randn(4,4) * 0.5

In [19]:
h = torch.zeros(4)                        # start over with a blank hidden state
for i, x in zip(ids, words):
    h = torch.tanh(W_x @ x + W_h @ h)     # new word + old h, then overwrite h
    print(repr(tokenizer.decode([i])), h)

' always' tensor([-0.5675,  0.8702, -0.0696,  0.9145], grad_fn=<TanhBackward0>)
' thought' tensor([-0.3204, -0.9948,  1.0000, -0.5685], grad_fn=<TanhBackward0>)
' Jack' tensor([ 0.9951, -0.9999,  0.5891, -0.9981], grad_fn=<TanhBackward0>)


In [28]:
def rnn(ids):
    h = torch.zeros(4)
    for x in embedding(torch.tensor(ids)):
        h = torch.tanh(W_x @ x + W_h @ h)
    return h

In [29]:
print(rnn(ids))

tensor([0.9971, 0.1653, 0.9957, 0.3751], grad_fn=<TanhBackward0>)


In [30]:
all_ids = tokenizer.encode(raw_text)
swap = tokenizer.encode(" banana")[0]

for n in [3, 10, 50]:
    s = all_ids[3:3+n]                    # n tokens starting at " always"
    base  = rnn(s)
    first = rnn([swap] + s[1:])           # change only the first word
    last  = rnn(s[:-1] + [swap])          # change only the last word
    print(n, (first - base).abs().sum().item(), (last - base).abs().sum().item())

3 1.685612678527832 1.588960886001587
10 0.0334036648273468 3.933704376220703
50 0.0 5.293532371520996
